# 🏦 Credit Risk Prediction — Full ML Pipeline
**Đồ án Máy học | LightGBM vs Random Forest vs Logistic Regression**

Dataset: `Data/credit_risk_dataset.csv` (32,581 mẫu, 12 cột: 11 features + 1 target)  
Bài toán: Phân loại nhị phân — dự đoán khả năng vỡ nợ (`loan_status`)

---
**Mục lục**
1. Load & Inspect Data  
2. Exploratory Data Analysis (EDA)  
3. Preprocessing  
4. Training: Logistic Regression / Random Forest / LightGBM  
5. Evaluation & So sánh  
6. Feature Importance & SHAP  
7. Tổng kết


## 1. Setup & Load Data

In [ ]:
%pip install lightgbm shap scikit-learn pandas numpy matplotlib seaborn jinja2 -q

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import shap
import time
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report
)
import lightgbm as lgb

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

try:
    display
except NameError:
    def display(obj):
        print(obj)

# ── Style ──────────────────────────────────────────────────
PALETTE = ["#4C72B0", "#55A868", "#C44E52"]
BLUE, GREEN, RED = PALETTE
BG = "#F8F9FA"
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor":   BG,
    "axes.spines.top":  False,
    "axes.spines.right": False,
})

print("✅ Libraries loaded")

In [ ]:
BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
data_candidates = [
    BASE_DIR / "Data" / "credit_risk_dataset.csv",
    BASE_DIR / "credit_risk_dataset.csv",
    Path.cwd() / "Data" / "credit_risk_dataset.csv",
    Path.cwd() / "credit_risk_dataset.csv",
]

for data_path in data_candidates:
    if data_path.exists():
        break
else:
    raise FileNotFoundError(
        "Không tìm thấy credit_risk_dataset.csv. "
        "Hãy đặt file trong cùng thư mục với mayhoc.py hoặc trong thư mục Data/."
    )


In [ ]:
df = pd.read_csv(data_path)
print(f"Data path: {data_path}")

print(f"Shape  : {df.shape}")
print(f"Columns: {df.columns.tolist()}\n")
print(df.dtypes)
print("\nFirst 5 rows:")
df.head()

In [ ]:
print("=== Missing Values ===")
print(df.isnull().sum())
print(f"\nTotal missing: {df.isnull().sum().sum()}")
print("\n=== Target Distribution ===")
print(df['loan_status'].value_counts())
print(f"\nClass ratio: {(df['loan_status']==0).sum()} không vỡ nợ | {(df['loan_status']==1).sum()} vỡ nợ")

### 1.1 Nhận diện biến & kiểm tra chất lượng dữ liệu
Theo Chương 3, trước khi làm sạch cần nhận diện loại biến, mức thiếu hụt, trùng lặp và miền giá trị để chọn cách xử lý phù hợp.

In [ ]:
target_col = "loan_status"
num_cols = ["person_age", "person_income", "loan_amnt", "loan_int_rate",
            "person_emp_length", "loan_percent_income", "cb_person_cred_hist_length"]
cat_cols = ["person_home_ownership", "loan_intent", "loan_grade", "cb_person_default_on_file"]

feature_roles = {
    "person_age": "numeric/raw",
    "person_income": "numeric/raw",
    "person_home_ownership": "categorical/nominal",
    "person_emp_length": "numeric/raw",
    "loan_intent": "categorical/nominal",
    "loan_grade": "categorical/ordinal-like",
    "loan_amnt": "numeric/raw",
    "loan_int_rate": "numeric/raw",
    "loan_percent_income": "numeric/derived",
    "cb_person_default_on_file": "categorical/binary",
    "cb_person_cred_hist_length": "numeric/raw",
    "loan_status": "target/binary",
}

data_quality = pd.DataFrame({
    "role": [feature_roles.get(col, "unknown") for col in df.columns],
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(dropna=False),
})

display(data_quality)
print(f"Duplicate rows: {df.duplicated().sum()}")
print()
print("Categorical domains:")
for col in cat_cols:
    print(f"- {col}: {sorted(df[col].dropna().unique().tolist())}")

## 2. Exploratory Data Analysis (EDA)

### 2.1 Phân phối biến mục tiêu `loan_status`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), facecolor=BG)
fig.suptitle("Phân phối biến mục tiêu — loan_status", fontsize=14, fontweight="bold", y=1.02)

counts = df["loan_status"].value_counts()
labels = ["Không vỡ nợ (0)", "Vỡ nợ (1)"]
colors_pie = ["#4C72B0", "#C44E52"]

axes[0].pie(
    counts,
    labels=labels,
    autopct="%1.1f%%",
    startangle=90,
    colors=colors_pie,
    wedgeprops=dict(edgecolor="white", linewidth=2),
)
axes[0].set_title("Tỷ lệ (%)")

bars = axes[1].bar(
    labels,
    counts.values,
    color=colors_pie,
    edgecolor="white",
    linewidth=1.5,
    width=0.5,
)

for bar, val in zip(bars, counts.values):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 200,
        f"{val:,}",
        ha="center",
        va="bottom",
        fontweight="bold",
        fontsize=12,
    )

axes[1].set_ylabel("Số lượng mẫu")
axes[1].set_title("Số lượng")
axes[1].set_ylim(0, counts.max() * 1.15)

plt.tight_layout()
plt.show()

print("⚠️  Class imbalance: ~78% không vỡ nợ vs ~22% vỡ nợ → cần xử lý khi training")

### 2.2 Thống kê mô tả Numeric Features
Dùng các thước đo trung tâm và phân tán để nhìn nhanh độ lệch, độ rộng và dấu hiệu bất thường của từng biến định lượng.

In [ ]:
desc_stats = df[num_cols].agg(["count", "mean", "median", "std", "min", "max"]).T
desc_stats["q1"] = df[num_cols].quantile(0.25)
desc_stats["q3"] = df[num_cols].quantile(0.75)
desc_stats["iqr"] = desc_stats["q3"] - desc_stats["q1"]
desc_stats["missing_pct"] = (df[num_cols].isna().mean() * 100).round(2)

desc_stats = desc_stats[["count", "missing_pct", "mean", "median", "std", "min", "q1", "q3", "max", "iqr"]].round(3)
display(desc_stats)

### 2.3 Phân phối Numeric Features theo class

In [ ]:
num_cols = ["person_age", "person_income", "loan_amnt", "loan_int_rate",
            "person_emp_length", "loan_percent_income", "cb_person_cred_hist_length"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8), facecolor=BG)
fig.suptitle("Phân phối Numeric Features (theo loan_status)", fontsize=14, fontweight="bold")
axes = axes.flatten()

for i, col in enumerate(num_cols):
    data0 = df[df["loan_status"] == 0][col].dropna()
    data1 = df[df["loan_status"] == 1][col].dropna()
    axes[i].hist(data0, bins=35, alpha=0.6, color=BLUE,  label="Không vỡ nợ", density=True)
    axes[i].hist(data1, bins=35, alpha=0.6, color=RED,   label="Vỡ nợ",       density=True)
    axes[i].set_title(col, fontsize=11, fontweight="bold")
    axes[i].legend(fontsize=8)

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

### 2.4 Categorical Features

In [ ]:
cat_cols = ["person_home_ownership", "loan_intent", "loan_grade", "cb_person_default_on_file"]

fig, axes = plt.subplots(1, 4, figsize=(18, 5), facecolor=BG)
fig.suptitle("Phân phối Categorical Features (theo loan_status)", fontsize=14, fontweight="bold")

for ax, col in zip(axes, cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, hue="loan_status", ax=ax, order=order,
                  palette={0: BLUE, 1: RED})
    ax.set_title(col, fontsize=11, fontweight="bold")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
    ax.get_legend().remove() # Ẩn legend riêng lẻ của từng ô con

# Tạo 1 legend chung duy nhất cho toàn bộ bức tranh ở góc trên bên phải
fig.legend(labels=["Không vỡ nợ", "Vỡ nợ"], loc="upper right", 
           bbox_to_anchor=(0.98, 0.98), title="Trạng thái khoản vay")

plt.tight_layout()
plt.show()

### 2.5 Correlation Heatmap

In [ ]:
# 1. Định nghĩa lại danh sách các cột (Bổ sung bước này để tránh lỗi NameError)
num_cols = ["person_age", "person_income", "loan_amnt", "loan_int_rate",
            "person_emp_length", "loan_percent_income", "cb_person_cred_hist_length"]

cat_cols = ["person_home_ownership", "loan_intent", "loan_grade", "cb_person_default_on_file"]

# 2. Tạo một bản copy dữ liệu để tính toán tương quan (bao gồm cả biến chữ đã mã hóa)
df_corr_calc = df.copy()

# Mã hóa các cột chữ thành số (cat.codes) để đưa vào ma trận tương quan tự động
for col in cat_cols:
    df_corr_calc[col] = df_corr_calc[col].astype('category').cat.codes

# 3. Tính ma trận tương quan trên tất cả các biến liên quan
all_features = num_cols + cat_cols
corr_matrix = df_corr_calc[all_features + ["loan_status"]].corr()

# 4. Tự động lấy ra độ tương quan của TẤT CẢ các biến đối với 'loan_status' (bỏ chính nó)
target_corr = corr_matrix["loan_status"].drop("loan_status")

# 5. Tìm 2 biến có mức độ tương quan mạnh nhất (dựa trên trị tuyệt đối để lấy cả thuận lẫn nghịch)
top_features = target_corr.abs().nlargest(2).index.tolist()

# Lấy giá trị tương quan thực tế từ ma trận của 2 biến này
corr_val1 = corr_matrix.loc[top_features[0], "loan_status"]
corr_val2 = corr_matrix.loc[top_features[1], "loan_status"]

# 6. VẼ BIỂU ĐỒ HEATMAP TOÀN DIỆN
fig, ax = plt.subplots(figsize=(11, 8), facecolor=BG)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=ax,
            annot_kws={"size": 10}, cbar_kws={"shrink": 0.8})
ax.set_title("Ma trận tương quan toàn diện (Sinh tự động)", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

# 7. IN KẾT LUẬN TỰ ĐỘNG TỪ KẾT QUẢ CHẠY THỰC TẾ
print(f"Hai biến có độ tương quan cao nhất với loan_status là:")
print(f"   1. Khía cạnh '{top_features[0]}' với hệ số tương quan thực tế: {corr_val1:.2f}")
print(f"   2. Khía cạnh '{top_features[1]}' với hệ số tương quan thực tế: {corr_val2:.2f}")

### 2.6 Phát hiện Outliers

In [ ]:
q1 = df[num_cols].quantile(0.25)
q3 = df[num_cols].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

iqr_outlier_mask = (df[num_cols].lt(lower)) | (df[num_cols].gt(upper))
outlier_report = pd.DataFrame({
    "lower_iqr": lower,
    "upper_iqr": upper,
    "outlier_count": iqr_outlier_mask.sum(),
    "outlier_pct": (iqr_outlier_mask.mean() * 100).round(2),
    "min": df[num_cols].min(),
    "max": df[num_cols].max(),
}).round(3)

display(outlier_report)

fig, axes = plt.subplots(2, 4, figsize=(18, 8), facecolor=BG)
fig.suptitle("Boxplot Numeric Features — kiểm tra outlier", fontsize=14, fontweight="bold")
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(y=df[col], ax=axes[i], color=BLUE, width=0.45, fliersize=2)
    axes[i].set_title(col, fontsize=11, fontweight="bold")
    axes[i].set_xlabel("")

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

age_outliers = (df["person_age"] > 100).sum()
emp_outliers = ((df["person_emp_length"].notna()) & (df["person_emp_length"] > 60)).sum()
print("Domain-rule anomalies cần loại bỏ:")
print(f"- person_age > 100: {age_outliers} rows")
print(f"- person_emp_length > 60: {emp_outliers} rows")
print()
print("Quyết định xử lý: không loại toàn bộ IQR outliers vì thu nhập/khoản vay cao có thể hợp lệ; chỉ loại giá trị bất khả lý theo miền dữ liệu.")

## 3. Preprocessing

In [ ]:
# ─── 3.1 Làm sạch dữ liệu: duplicate + domain-rule outliers ───
rows_before = len(df)
df_clean = df.drop_duplicates().copy()
duplicate_removed = rows_before - len(df_clean)

age_outliers = (df_clean["person_age"] > 100).sum()
emp_outliers = ((df_clean["person_emp_length"].notna()) & (df_clean["person_emp_length"] > 60)).sum()

df_clean = df_clean[
    (df_clean["person_age"] <= 100) &
    (df_clean["person_emp_length"].isna() | (df_clean["person_emp_length"] <= 60))
].copy()

print(f"✅ Duplicate removed: {duplicate_removed} rows")
print(f"✅ Domain-rule outliers removed: {age_outliers + emp_outliers} rows")
print(f"✅ Cleaned data còn lại: {len(df_clean)} rows")
print()
print("Missing values sau bước làm sạch:")
print(df_clean[["person_emp_length", "loan_int_rate"]].isna().sum())

# ─── 3.2 Tách X, y và Chia Train/Test TRƯỚC để chống Data Leakage ───
X = df_clean.drop("loan_status", axis=1)
y = df_clean["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"✅ Train/Test Split xong. Train: {X_train.shape} | Test: {X_test.shape}")

# Class imbalance được xử lý chủ yếu bằng threshold tuning trên validation set.


# ─── MÔ HÌNH 1: CHUẨN BỊ DỮ LIỆU CHO LIGHTGBM ───
cat_cols = ["person_home_ownership", "loan_intent", "loan_grade", "cb_person_default_on_file"]
lgb_cat_cols = cat_cols


def add_lgbm_features(X_input):
    X_out = X_input.copy()
    grade_map = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}

    X_out["loan_int_rate_missing"] = X_out["loan_int_rate"].isna().astype("int8")
    X_out["person_emp_length_missing"] = X_out["person_emp_length"].isna().astype("int8")
    X_out["person_income_log"] = np.log1p(X_out["person_income"])
    X_out["loan_amnt_log"] = np.log1p(X_out["loan_amnt"])
    X_out["income_to_loan"] = X_out["person_income"] / (X_out["loan_amnt"] + 1)
    X_out["loan_to_income_check"] = X_out["loan_amnt"] / (X_out["person_income"] + 1)
    X_out["rate_x_percent_income"] = X_out["loan_int_rate"] * X_out["loan_percent_income"]
    X_out["emp_to_age"] = X_out["person_emp_length"] / (X_out["person_age"] + 1)
    X_out["cred_hist_to_age"] = X_out["cb_person_cred_hist_length"] / (X_out["person_age"] + 1)
    X_out["loan_grade_ord"] = X_out["loan_grade"].map(grade_map).astype(float)
    X_out["high_loan_percent_income"] = (X_out["loan_percent_income"] >= 0.35).astype("int8")
    X_out["very_high_int_rate"] = (X_out["loan_int_rate"] >= 15).astype("int8")
    X_out["young_high_loan"] = (
        (X_out["person_age"] < 30) & (X_out["loan_percent_income"] >= 0.25)
    ).astype("int8")
    X_out["rate_x_grade"] = X_out["loan_int_rate"] * X_out["loan_grade_ord"]
    X_out["grade_x_percent_income"] = X_out["loan_grade_ord"] * X_out["loan_percent_income"]
    X_out["default_file_x_grade"] = (
        (X_out["cb_person_default_on_file"] == "Y").astype("int8") * X_out["loan_grade_ord"]
    )
    X_out["rent_high_percent_income"] = (
        (X_out["person_home_ownership"] == "RENT") &
        (X_out["loan_percent_income"] >= 0.30)
    ).astype("int8")
    X_out["own_low_percent_income"] = (
        (X_out["person_home_ownership"] == "OWN") &
        (X_out["loan_percent_income"] < 0.20)
    ).astype("int8")
    X_out["debt_consolidation_high_rate"] = (
        (X_out["loan_intent"] == "DEBTCONSOLIDATION") &
        (X_out["loan_int_rate"] >= 14)
    ).astype("int8")
    X_out["medical_high_percent_income"] = (
        (X_out["loan_intent"] == "MEDICAL") &
        (X_out["loan_percent_income"] >= 0.25)
    ).astype("int8")
    X_out["credit_history_short"] = (X_out["cb_person_cred_hist_length"] <= 3).astype("int8")

    return X_out


def align_lgb_categories(X_train_part, X_other_part, categorical_cols):
    X_train_part = X_train_part.copy()
    X_other_part = X_other_part.copy()
    for col in categorical_cols:
        X_train_part[col] = X_train_part[col].astype("category")
        X_other_part[col] = pd.Categorical(X_other_part[col], categories=X_train_part[col].cat.categories)
    return X_train_part, X_other_part


def find_best_f1_threshold(
    y_true,
    y_prob,
    start=0.05,
    stop=0.95,
    step=0.005,
    tie_tolerance=0.0,
    prefer_high_threshold=False,
):
    thresholds = np.arange(start, stop + step, step)
    f1_scores = np.array([f1_score(y_true, y_prob >= thr) for thr in thresholds])
    if tie_tolerance > 0:
        best_score = f1_scores.max()
        candidate_idxs = np.flatnonzero(f1_scores >= best_score - tie_tolerance)
        best_idx = int(candidate_idxs[-1] if prefer_high_threshold else candidate_idxs[0])
    else:
        best_idx = int(f1_scores.argmax())
    return float(thresholds[best_idx]), float(f1_scores[best_idx])


X_train_lgb = add_lgbm_features(X_train)
X_test_lgb = add_lgbm_features(X_test)
X_train_lgb, X_test_lgb = align_lgb_categories(X_train_lgb, X_test_lgb, lgb_cat_cols)
print("✅ Dữ liệu cho LightGBM: Feature engineering + category alignment + giữ nguyên NaN")


# ─── MÔ HÌNH 2 & 3: CHUẨN BỊ CHO RANDOM FOREST & LOGISTIC REGRESSION ───
# Bước A: Thêm missing indicators rồi impute dựa trên TRAIN để tránh data leakage
X_train_imputed = X_train.copy()
X_test_imputed = X_test.copy()

for col in ["loan_int_rate", "person_emp_length"]:
    X_train_imputed[f"{col}_missing"] = X_train_imputed[col].isna().astype(int)
    X_test_imputed[f"{col}_missing"] = X_test_imputed[col].isna().astype(int)

# Điền loan_int_rate theo median của loan_grade trong tập TRAIN; fallback bằng global median nếu cần
loan_grade_medians = X_train_imputed.groupby("loan_grade")["loan_int_rate"].median()
global_loan_int_median = X_train_imputed["loan_int_rate"].median()
X_train_imputed["loan_int_rate"] = (
    X_train_imputed["loan_int_rate"]
    .fillna(X_train_imputed["loan_grade"].map(loan_grade_medians))
    .fillna(global_loan_int_median)
)
X_test_imputed["loan_int_rate"] = (
    X_test_imputed["loan_int_rate"]
    .fillna(X_test_imputed["loan_grade"].map(loan_grade_medians))
    .fillna(global_loan_int_median)
)

# Điền person_emp_length theo median của tập TRAIN
emp_median = X_train_imputed["person_emp_length"].median()
X_train_imputed["person_emp_length"] = X_train_imputed["person_emp_length"].fillna(emp_median)
X_test_imputed["person_emp_length"] = X_test_imputed["person_emp_length"].fillna(emp_median)

# Bước B: Mã hóa phù hợp cho Random Forest & Logistic Regression
X_train_encoded = pd.get_dummies(X_train_imputed, columns=cat_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_imputed, columns=cat_cols, drop_first=True)

# Đảm bảo tập test có số cột trùng khớp tập train sau khi One-Hot
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)
print("✅ Dữ liệu cho RF & LR: Missing indicators + imputation + One-Hot Encoding")

# Bước C: Scale riêng cho Logistic Regression
scaler = StandardScaler()
X_train_lr_scaled = scaler.fit_transform(X_train_encoded)
X_test_lr_scaled = scaler.transform(X_test_encoded)
print("✅ Dữ liệu cho Logistic Regression: Đã được Scale")

### 3.2 Kiểm tra dữ liệu sau tiền xử lý
Kiểm tra lại các điều kiện quan trọng sau làm sạch, chia dữ liệu, imputation và encoding để bảo đảm dữ liệu đầu vào cho mô hình nhất quán.

In [ ]:
post_check = pd.DataFrame({
    "check": [
        "Rows after cleaning",
        "Duplicate rows",
        "person_age > 100",
        "person_emp_length > 60",
        "Missing loan_int_rate retained before model-specific handling",
        "Missing person_emp_length retained before model-specific handling",
        "NaN in RF/LR train after imputation",
        "NaN in RF/LR test after imputation",
        "Encoded feature count for RF/LR",
    ],
    "value": [
        len(df_clean),
        df_clean.duplicated().sum(),
        (df_clean["person_age"] > 100).sum(),
        ((df_clean["person_emp_length"].notna()) & (df_clean["person_emp_length"] > 60)).sum(),
        X_train_lgb["loan_int_rate"].isna().sum() + X_test_lgb["loan_int_rate"].isna().sum(),
        X_train_lgb["person_emp_length"].isna().sum() + X_test_lgb["person_emp_length"].isna().sum(),
        X_train_encoded.isna().sum().sum(),
        X_test_encoded.isna().sum().sum(),
        X_train_encoded.shape[1],
    ]
})

display(post_check)

split_check = pd.DataFrame({
    "dataset": ["Full cleaned", "Train", "Test"],
    "rows": [len(df_clean), len(y_train), len(y_test)],
    "default_rate": [y.mean(), y_train.mean(), y_test.mean()],
})
split_check["default_rate"] = (split_check["default_rate"] * 100).round(2)
display(split_check)

assert df_clean.duplicated().sum() == 0
assert (df_clean["person_age"] > 100).sum() == 0
assert ((df_clean["person_emp_length"].notna()) & (df_clean["person_emp_length"] > 60)).sum() == 0
assert X_train_encoded.isna().sum().sum() == 0
assert X_test_encoded.isna().sum().sum() == 0
print("✅ Post-preprocessing checks passed")

## 4. Training Models

### 4.1 Baseline 1 — Logistic Regression

In [ ]:
t0 = time.time()
# Khởi tạo mô hình
lr = LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced", random_state=42)

# SỬA TÊN BIẾN: Thay X_train_s bằng X_train_lr_scaled
lr.fit(X_train_lr_scaled, y_train)
t_lr = time.time() - t0

# SỬA TÊN BIẾN: Thay X_test_s bằng X_test_lr_scaled
lr_pred = lr.predict(X_test_lr_scaled)
lr_prob = lr.predict_proba(X_test_lr_scaled)[:, 1]

print(f"✅ Logistic Regression trained in {t_lr:.3f}s")
print("\nClassification Report:")
print(classification_report(y_test, lr_pred, target_names=["Không vỡ nợ", "Vỡ nợ"]))

### 4.2 Baseline 2 — Random Forest

In [ ]:
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=200, 
    max_depth=12,
    class_weight="balanced", 
    random_state=42, 
    n_jobs=-1
)

# SỬA TÊN BIẾN: Dùng X_train_encoded đã được xử lý chữ và NaN
rf.fit(X_train_encoded, y_train)
t_rf = time.time() - t0

# SỬA TÊN BIẾN: Dùng X_test_encoded
rf_pred = rf.predict(X_test_encoded)
rf_prob = rf.predict_proba(X_test_encoded)[:, 1]

print(f"✅ Random Forest trained in {t_rf:.3f}s")
print("\nClassification Report:")
print(classification_report(y_test, rf_pred, target_names=["Không vỡ nợ", "Vỡ nợ"]))

### 4.3 LightGBM ⭐

In [ ]:
t0 = time.time()

X_lgb_fit, X_lgb_valid, y_lgb_fit, y_lgb_valid = train_test_split(
    X_train_lgb, y_train, test_size=0.2, random_state=42, stratify=y_train
)

lgbm = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=2500,
    learning_rate=0.015,
    num_leaves=31,
    max_depth=-1,
    scale_pos_weight=1.0,
    subsample=0.8,
    colsample_bytree=0.75,
    min_child_samples=10,
    min_split_gain=0.0,
    reg_alpha=0.0,
    reg_lambda=0.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgbm.fit(
    X_lgb_fit, y_lgb_fit,
    eval_set=[(X_lgb_valid, y_lgb_valid)],
    categorical_feature=lgb_cat_cols,
    callbacks=[
        lgb.early_stopping(80, verbose=False),
        lgb.log_evaluation(-1)
    ]
)
t_lgbm = time.time() - t0

lgbm_valid_prob = lgbm.predict_proba(X_lgb_valid)[:, 1]
lgbm_best_threshold, lgbm_valid_f1 = find_best_f1_threshold(
    y_lgb_valid,
    lgbm_valid_prob,
    tie_tolerance=0.0005,
    prefer_high_threshold=True,
)
lgbm_prob = lgbm.predict_proba(X_test_lgb)[:, 1]
lgbm_pred = (lgbm_prob >= lgbm_best_threshold).astype(int)
lgbm_test_oracle_threshold, lgbm_test_oracle_f1 = find_best_f1_threshold(y_test, lgbm_prob)

print(f"✅ LightGBM trained in {t_lgbm:.3f}s | Best iteration: {lgbm.best_iteration_}")
print("Validation set được tách từ training set; test set chỉ dùng cho đánh giá cuối.")
print(f"✅ Best F1 threshold trên validation: {lgbm_best_threshold:.3f} | Validation F1: {lgbm_valid_f1:.4f}")
print(
    f"ℹ️ Oracle threshold trên test (chỉ để chẩn đoán, KHÔNG dùng làm dự báo chính): "
    f"{lgbm_test_oracle_threshold:.3f} | Oracle Test F1: {lgbm_test_oracle_f1:.4f}"
)
print()
print("Classification Report:")
print(classification_report(y_test, lgbm_pred, target_names=["Không vỡ nợ", "Vỡ nợ"]))

## 5. Evaluation & So sánh

### 5.1 Bảng tổng hợp metrics

In [ ]:
results = {
    "Logistic Regression": dict(pred=lr_pred,    prob=lr_prob,    time=t_lr),
    "Random Forest":       dict(pred=rf_pred,    prob=rf_prob,    time=t_rf),
    "LightGBM":            dict(pred=lgbm_pred,  prob=lgbm_prob,  time=t_lgbm),
}

rows = []
for name, v in results.items():
    rows.append({
        "Model":     name,
        "Accuracy":  round(accuracy_score(y_test, v["pred"]), 4),
        "Precision": round(precision_score(y_test, v["pred"]), 4),
        "Recall":    round(recall_score(y_test, v["pred"]), 4),
        "F1-score":  round(f1_score(y_test, v["pred"]), 4),
        "AUC-ROC":   round(roc_auc_score(y_test, v["prob"]), 4),
        "Time (s)":  round(v["time"], 3),
    })

df_metrics = pd.DataFrame(rows).set_index("Model")
lightgbm_f1 = df_metrics.loc["LightGBM", "F1-score"]
target_f1 = 0.95
if lightgbm_f1 >= target_f1:
    print(f"✅ LightGBM đã đạt mục tiêu F1-score >= {target_f1:.2f}: {lightgbm_f1:.4f}")
else:
    print(
        f"⚠️ LightGBM F1-score sạch hiện tại là {lightgbm_f1:.4f}, chưa đạt mục tiêu {target_f1:.2f}. "
        f"Ngay cả khi tune threshold trực tiếp trên test set (không hợp lệ để báo cáo chính), "
        f"F1 cao nhất quan sát được cũng chỉ khoảng {lgbm_test_oracle_f1:.4f}. "
        "Vì vậy mốc 0.95 không thực tế với split/test sạch hiện tại nếu không dùng data leakage "
        "hoặc đánh giá trên chính dữ liệu train."
    )

df_metrics.style.background_gradient(cmap="YlGn", subset=["Accuracy","F1-score","AUC-ROC"]) \
                .bar(subset=["Time (s)"], color="#C44E52", vmin=0)

### 5.2 Biểu đồ so sánh Metrics

In [ ]:
metric_cols = ["Accuracy", "Precision", "Recall", "F1-score", "AUC-ROC"]
fig, ax = plt.subplots(figsize=(13, 5.5), facecolor=BG)
x     = np.arange(len(metric_cols))
width = 0.25

for i, (model, color) in enumerate(zip(df_metrics.index, PALETTE)):
    vals = df_metrics.loc[model, metric_cols].values.astype(float)
    bars = ax.bar(x + i*width, vals, width, label=model, color=color,
                  edgecolor="white", linewidth=1.2)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f"{v:.3f}", ha="center", va="bottom", fontsize=8.5, fontweight="bold")

ax.set_xticks(x + width)
ax.set_xticklabels(metric_cols, fontsize=11)
ax.set_ylim(0, 1.13)
ax.set_ylabel("Score")
ax.set_title("So sánh hiệu năng — 3 mô hình", fontsize=13, fontweight="bold", pad=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### 5.3 ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6.5), facecolor=BG)
for (name, v), color in zip(results.items(), PALETTE):
    fpr, tpr, _ = roc_curve(y_test, v["prob"])
    auc = roc_auc_score(y_test, v["prob"])
    ax.plot(fpr, tpr, lw=2.5, color=color, label=f"{name}  (AUC = {auc:.4f})")

ax.plot([0,1],[0,1], "k--", lw=1.2, alpha=0.5, label="Random baseline")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curve — So sánh 3 mô hình", fontsize=13, fontweight="bold", pad=12)
ax.legend(fontsize=10, loc="lower right")
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.05])
plt.tight_layout()
plt.show()

### 5.4 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), facecolor=BG)
fig.suptitle("Confusion Matrix — 3 mô hình", fontsize=14, fontweight="bold", y=1.02)

for ax, (name, v), color in zip(axes, results.items(), PALETTE):
    cm  = confusion_matrix(y_test, v["pred"])
    pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    annot = [[f"{cm[i,j]}\n({pct[i,j]:.1f}%)" for j in range(2)] for i in range(2)]
    cmap  = sns.light_palette(color, as_cmap=True)
    sns.heatmap(cm, annot=annot, fmt="", cmap=cmap, ax=ax,
                xticklabels=["Pred: 0","Pred: 1"],
                yticklabels=["Actual: 0","Actual: 1"],
                linewidths=1, cbar=False, annot_kws={"size": 12})
    ax.set_title(name, fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

### 5.5 Training Time

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5), facecolor=BG)
models = list(results.keys())
times  = [results[m]["time"] for m in models]
bars   = ax.bar(models, times, color=PALETTE, edgecolor="white", linewidth=1.5, width=0.5)
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{t:.2f}s", ha="center", va="bottom", fontweight="bold", fontsize=13)
ax.set_ylabel("Thời gian huấn luyện (giây)", fontsize=11)
ax.set_title("So sánh tốc độ huấn luyện", fontsize=13, fontweight="bold", pad=12)
ax.set_ylim(0, max(times) * 1.3)
plt.tight_layout()
plt.show()

## 6. Feature Importance & SHAP (LightGBM)

### 6.1 Feature Importance (split count)

In [ ]:
# 1. Tạo Series độ quan trọng gắn liền với tên cột của tập huấn luyện LightGBM (để đảm bảo an toàn tuyệt đối)
fi = pd.Series(lgbm.feature_importances_, index=X_train_lgb.columns).sort_values(ascending=True)

# 2. Tự động xác định danh sách TOP 2 features quan trọng nhất từ kết quả chạy
top_2_features = fi.nlargest(2).index.tolist()

# 3. Cấu hình màu sắc ĐỘNG: Tô màu ĐỎ cho cả 2 features nằm trong TOP 2, các cột còn lại màu XANH
colors_fi = [RED if c in top_2_features else BLUE for c in fi.index]

# 4. VẼ BIỂU ĐỒ BARH
fig, ax = plt.subplots(figsize=(9, 6), facecolor=BG)
bars = ax.barh(fi.index, fi.values, color=colors_fi, edgecolor="white", linewidth=0.8)

# Điền số lượng split cụ thể lên đầu mỗi thanh bar
for bar, val in zip(bars, fi.values):
    ax.text(bar.get_width() + fi.max() * 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:,.0f}", va="center", fontsize=9, fontweight="bold")

ax.set_xlabel("Importance (split count)", fontsize=11)
ax.set_title("LightGBM — Feature Importance (Tự động cập nhật)", fontsize=13, fontweight="bold", pad=12)
ax.set_xlim(0, fi.max() * 1.15)
plt.tight_layout()
plt.show()

# 5. IN KẾT LUẬN TỰ ĐỘNG TỪ KẾT QUẢ MÔ HÌNH CHẠY
print(f"💡 Kết luận tự động từ mô hình LightGBM:")
print(f"   - Hai thuộc tính quan trọng nhất quyết định phân nhánh cây là: '{top_2_features[0]}' (Top 1) và '{top_2_features[1]}' (Top 2).")
print(f"   - Chỉ số quan trọng (Split count) lần lượt là: {fi[top_2_features[0]]} và {fi[top_2_features[1]]}.")

### 6.2 SHAP Summary Plot

In [ ]:
print("Computing SHAP values (sample 1000)...")
explainer   = shap.TreeExplainer(lgbm)

# SỬA TÊN BIẾN: Lấy mẫu từ tập X_test_lgb chuyên dụng của LightGBM
sample_idx  = X_test_lgb.sample(1000, random_state=42).index
shap_values = explainer.shap_values(X_test_lgb.loc[sample_idx])

if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

plt.figure(figsize=(10, 6), facecolor=BG)
# SỬA TÊN BIẾN: Truyền dữ liệu X_test_lgb vào summary_plot
shap.summary_plot(sv, X_test_lgb.loc[sample_idx], show=False)
plt.title("SHAP Summary Plot — LightGBM (1000 samples)", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

print("\n💡 Đọc biểu đồ:")
print("  - Màu đỏ = giá trị feature cao | Màu xanh = giá trị feature thấp")
print("  - SHAP value > 0 → feature đẩy prediction về phía VỠ NỢ (class 1)")

### 6.3 SHAP Waterfall — Giải thích 1 prediction cụ thể

In [ ]:
# 1. Tìm chính xác nhãn index gốc của một mẫu VỠ NỢ (y_test == 1) từ tập Test
idx_default = y_test[y_test == 1].index[0]

# 2. Khởi tạo bộ giải thích TreeExplainer
explainer = shap.TreeExplainer(lgbm)

# 3. Tính toán SHAP value trên đúng dòng dữ liệu LightGBM
explanation = explainer(X_test_lgb.loc[[idx_default]])
default_prob = lgbm.predict_proba(X_test_lgb.loc[[idx_default]])[0, 1]
raw_margin = explanation.values[0].sum() + explainer.expected_value

# 4. VẼ BIỂU ĐỒ WATERFALL
plt.figure(figsize=(12, 6), facecolor=BG)
shap.plots.waterfall(explanation[0], show=False)
plt.title(f"SHAP Waterfall — Giải thích hồ sơ VỠ NỢ số #{idx_default}", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

# 5. IN KẾT LUẬN ĐỘNG TRỰC TIẾP TỪ BIẾN CỦA MẪU DỮ LIỆU ĐÓ
print(f"💡 Phân tích hồ sơ khách hàng #{idx_default}:")
print(f"   - Xác suất dự báo VỠ NỢ của mô hình LightGBM: {default_prob:.2%}")
print(f"   - Raw margin/log-odds dùng trong SHAP waterfall: {raw_margin:.2f}")
print("   - Các thanh màu ĐỎ (giá trị > 0) là các yếu tố đẩy dự báo về phía nguy cơ VỠ NỢ.")

## 7. Tổng kết

In [ ]:
print("=" * 60)
print("  BẢNG SO SÁNH CUỐI CÙNG")
print("=" * 60)
print(df_metrics.to_string())
print()

best_auc = df_metrics["AUC-ROC"].idxmax()
best_f1  = df_metrics["F1-score"].idxmax()
fastest  = df_metrics["Time (s)"].idxmin()

print(f"★ Best AUC-ROC : {best_auc} ({df_metrics.loc[best_auc,'AUC-ROC']:.4f})")
print(f"★ Best F1-score: {best_f1} ({df_metrics.loc[best_f1,'F1-score']:.4f})")
print(f"★ Fastest      : {fastest} ({df_metrics.loc[fastest,'Time (s)']:.3f}s)")
print()

# Tính toán động tỷ lệ thời gian giữa LightGBM và Random Forest để kết luận cho chuẩn số liệu
rf_time = df_metrics.loc["Random Forest", "Time (s)"]
lgb_time = df_metrics.loc["LightGBM", "Time (s)"]
time_ratio = lgb_time / rf_time if rf_time > 0 else 0

print("KẾT LUẬN BIỆN GIẢI TỪ DỮ LIỆU THỰC TẾ:")
print(f"  • {best_auc} đạt AUC-ROC cao nhất ({df_metrics.loc[best_auc,'AUC-ROC']:.4f}) và {best_f1} cho F1 tốt nhất ({df_metrics.loc[best_f1,'F1-score']:.4f}).")

if lgb_time < rf_time:
    print(f"  • LightGBM tối ưu hơn khi nhanh hơn Random Forest ~{rf_time/lgb_time:.1f}x trong khi cho kết quả tốt hơn.")
else:
    print(
        f"  • Dù LightGBM mất nhiều thời gian huấn luyện hơn Random Forest ~{time_ratio:.1f}x "
        f"(do cấu hình {lgbm.get_params()['n_estimators']} cây và {lgbm.get_params()['num_leaves']} lá), "
        "nhưng đổi lại chất lượng phân loại đạt độ chính xác cao nhất."
    )

print(f"  • Logistic Regression đơn giản, tốc độ nhanh nhất ({df_metrics.loc['Logistic Regression', 'Time (s)']}s), Recall cao nhưng Precision thấp.")
print("    → Phù hợp khi tổ chức tín dụng ưu tiên bắt tối đa case vỡ nợ, chấp nhận tỷ lệ báo động giả (False Positive) cao.")
print("  • LightGBM được cải thiện bằng interaction features và threshold tuning trên validation set, giúp cân bằng Precision/Recall mà không dùng nhãn test để chọn ngưỡng.")